# PROYECTO MÓDULO 9 CIENCIA DE DATOS

# Proyecto: Clasificador Inteligente de Imágenes de Ropa

# Reinaldo Elgueta A.

## LECCIÓN 1: En PDF

## Lección 2: Apache Spark - Introducción y Configuración


📌 Objetivo: Configurar Spark e iniciar el procesamiento distribuido de datos
masivos.


📍 Tareas a desarrollar:

- Configurar SparkContext y SparkSession en un entorno local o clúster.
- Cargar datos iniciales en RDDs y explorar acciones básicas (count,take).
- Validar conectividad con datasets masivos de prueba.
- Entregar un notebook con la configuración y prueba inicial.

In [1]:
#=====================================================================
#Configurar SparkContext y SparkSession en un entorno local o clúster.
#=====================================================================


# Instalación de PySpark
!pip install pyspark -q

# Importación y configuración
from pyspark.sql import SparkSession

# Crear la SparkSession (el motor principal)
spark = SparkSession.builder \
    .appName("RetailMax_Setup") \
    .getOrCreate()

# Configurar el SparkContext (necesario para trabajar con RDDs)
sc = spark.sparkContext

print(f"Versión de Spark: {spark.version}")

Versión de Spark: 4.0.2


In [2]:
#=====================================================================
# EXTRA: CREACIÓN DE DATOS
#=====================================================================

# Lista de 10,000 registros sintéticos de RetailMax
# Formato: (ID_Transaccion, ID_Usuario, Monto, Categoria)
import random

categorias = ["Electrónica", "Hogar", "Ropa", "Libros", "Juguetes"]
data_sintetica = [
    (i, f"user_{random.randint(1, 1000)}", round(random.uniform(10, 500), 2), random.choice(categorias))
    for i in range(10000)
]

# 2. Convertir la lista en un RDD (aquí ocurre la magia de Spark)
rdd_retail = sc.parallelize(data_sintetica)

print("¡Dataset sintético creado y cargado en el clúster!")

¡Dataset sintético creado y cargado en el clúster!


In [3]:
#=====================================================================
# Cargar datos iniciales en RDDs y explorar acciones básicas (count,take)
#=====================================================================

# Convertimos nuestra lista de transacciones en un RDD distribuido
rdd_transacciones = sc.parallelize(data_sintetica)

print("Ecosistema Big Data: Datos cargados en RDD exitosamente.")

#explorar acciones básicas (count,take)
# Acción A: count() -> Cuenta el número total de registros
total_registros = rdd_transacciones.count()
print(f"1. Acción COUNT: El dataset contiene {total_registros} transacciones.")

# Acción B: take(n) -> Extrae los primeros n elementos para inspección
muestra_datos = rdd_transacciones.take(5)
print("\n2. Acción TAKE (Primeros 5 registros):")
for registro in muestra_datos:
    print(registro)

# Acción C: first() -> Una variante rápida de take(1)
print(f"\n3. Acción FIRST: El primer registro es: {rdd_transacciones.first()}")

Ecosistema Big Data: Datos cargados en RDD exitosamente.
1. Acción COUNT: El dataset contiene 10000 transacciones.

2. Acción TAKE (Primeros 5 registros):
(0, 'user_58', 429.28, 'Juguetes')
(1, 'user_929', 217.71, 'Electrónica')
(2, 'user_783', 295.41, 'Juguetes')
(3, 'user_703', 101.55, 'Electrónica')
(4, 'user_786', 439.36, 'Ropa')

3. Acción FIRST: El primer registro es: (0, 'user_58', 429.28, 'Juguetes')


In [4]:
#=====================================================================
# Validar conectividad con datasets masivos de prueba.
#=====================================================================

# Tarea: Validar conectividad y estado del clúster
if not rdd_transacciones.isEmpty():
    print("\n[OK] Conectividad validada: El RDD no está vacío y responde a consultas.")
    print(f"[INFO] Particiones activas: {rdd_transacciones.getNumPartitions()}")
else:
    print("\n[ERROR] Fallo en la carga de datos.")


[OK] Conectividad validada: El RDD no está vacío y responde a consultas.
[INFO] Particiones activas: 2


## Lección 3: Elementos básicos de Spark (RDD, Transformaciones y Acciones)


📌 Objetivo: Manipular grandes volúmenes de datos mediante RDDs, aplicando transformaciones y acciones.

📍 Tareas a desarrollar:

- Crear RDDs y Pair RDDs a partir de datos de transacciones.
- Aplicar transformaciones (map, filter, flatMap, distinct, sortBy).
- Ejecutar acciones (collect, sum, mean, stdev) y documentar el linaje.
- Entregar notebook con código funcional y explicación del DAG
generado.

In [5]:
#=====================================================================
# Crear RDDs y Pair RDDs a partir de datos de transacciones.
#=====================================================================

# Se crea Pair RDD a partir de los datos (Estructura: Categoria, Monto)
# Recordamos que rdd_transacciones tiene: (ID, Usuario, Monto, Categoria)
rdd_transacciones = sc.parallelize(data_sintetica)

# Transformamos a Pair RDD: (Clave: Categoria, Valor: Monto)
pair_rdd = rdd_transacciones.map(lambda x: (x[3], x[2]))


#=====================================================================
# Aplicar transformaciones (map, filter, flatMap, distinct, sortBy).
#=====================================================================

# A. map: Ya lo usamos para el Pair RDD, pero hagamos otro para estandarizar categorías a mayúsculas
rdd_mapeado = pair_rdd.map(lambda x: (x[0].upper(), x[1]))

# B. filter: Filtrar solo ventas de la categoría 'ELECTRÓNICA' (o montos > 100)
rdd_filtrado = rdd_mapeado.filter(lambda x: x[1] > 100)

# C. flatMap: Útil para desglosar elementos. Aquí simulamos separar etiquetas de las categorías
rdd_flat = rdd_filtrado.flatMap(lambda x: (x[0], f"VALOR_{x[1]}"))

# D. distinct: Obtener las categorías únicas procesadas
rdd_unicos = rdd_mapeado.map(lambda x: x[0]).distinct()

# E. sortBy: Ordenar las transacciones filtradas por monto de mayor a menor
rdd_ordenado = rdd_filtrado.sortBy(lambda x: x[1], ascending=False)



#=====================================================================
# Ejecutar acciones (collect, sum, mean, stdev) y documentar el linaje.
#=====================================================================
print("\n")
# A. collect: Traer una muestra de los datos ordenados al driver
print("Muestra Ordenada (collect):", rdd_ordenado.take(5))

print("\n")
# B. sum, mean, stdev: Cálculos numéricos sobre los montos
montos = rdd_filtrado.map(lambda x: x[1])

print("\n")
print(f"Suma Total (sum): {montos.sum():.2f}")
print(f"Promedio (mean): {montos.mean():.2f}")
print(f"Desviación Estándar (stdev): {montos.stdev():.2f}")



#=====================================================================
# Entregar notebook con código funcional y explicación del DAG generado.
#=====================================================================

print("\n")
# C. Documentar el linaje (Lineage/DAG)
# Esta función muestra la secuencia de pasos que Spark seguirá
print("\n--- Linaje del RDD (DAG) ---")
print(rdd_ordenado.toDebugString().decode())

# EXPLICACIÓN : En documento PDF



Muestra Ordenada (collect): [('LIBROS', 499.98), ('JUGUETES', 499.86), ('LIBROS', 499.84), ('ELECTRÓNICA', 499.81), ('LIBROS', 499.81)]




Suma Total (sum): 2448845.31
Promedio (mean): 301.66
Desviación Estándar (stdev): 114.97



--- Linaje del RDD (DAG) ---
(2) PythonRDD[21] at RDD at PythonRDD.scala:56 []
 |  MapPartitionsRDD[16] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[15] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[14] at sortBy at /tmp/ipykernel_1203/1009149264.py:30 []
    |  PythonRDD[13] at sortBy at /tmp/ipykernel_1203/1009149264.py:30 []
    |  ParallelCollectionRDD[6] at readRDDFromFile at PythonRDD.scala:297 []


## Lección 4: Procesamiento de datos estructurados (Spark SQL y DataFrames)


📌 Objetivo: Procesar y analizar datos estructurados de manera optimizada
usando DataFrames y Spark SQL.


📍 Tareas a desarrollar:


- Transformar los RDDs generados en DataFrames y aplicar esquemas explícitos.
- Ejecutar consultas SQL para generar métricas de negocio (ventas por categoría, top productos).
- Guardar resultados en formato Parquet para la siguiente lección.
- Entregar notebook + archivo Parquet generado.

In [6]:

#=====================================================================
# Transformar los RDDs generados en DataFrames y aplicar esquemas explícitos.
#=====================================================================

from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

# Tarea: Definir el esquema explícito para RetailMax
schema = StructType([
    StructField("id_transaccion", IntegerType(), True),
    StructField("id_usuario", StringType(), True),
    StructField("monto", FloatType(), True),
    StructField("categoria", StringType(), True)
])

# Convertir el RDD de la lección anterior a DataFrame aplicando el esquema
df_retail = spark.createDataFrame(rdd_transacciones, schema)

# Mostrar la estructura para validar
df_retail.printSchema()
df_retail.show(5)




root
 |-- id_transaccion: integer (nullable = true)
 |-- id_usuario: string (nullable = true)
 |-- monto: float (nullable = true)
 |-- categoria: string (nullable = true)

+--------------+----------+------+-----------+
|id_transaccion|id_usuario| monto|  categoria|
+--------------+----------+------+-----------+
|             0|   user_58|429.28|   Juguetes|
|             1|  user_929|217.71|Electrónica|
|             2|  user_783|295.41|   Juguetes|
|             3|  user_703|101.55|Electrónica|
|             4|  user_786|439.36|       Ropa|
+--------------+----------+------+-----------+
only showing top 5 rows


In [7]:
#=====================================================================
# Ejecutar consultas SQL para generar métricas de negocio (ventas por categoría, top productos).
#=====================================================================

# Registrar como tabla temporal
df_retail.createOrReplaceTempView("ventas")

# Tarea: Ejecutar consultas SQL para generar métricas
# 1. Ventas totales por categoría
ventas_por_categoria = spark.sql("""
    SELECT categoria, SUM(monto) as total_ventas, COUNT(*) as cantidad_transacciones
    FROM ventas
    GROUP BY categoria
    ORDER BY total_ventas DESC
""")

# 2. Top productos/categorías más rentables
top_categorias = spark.sql("""
    SELECT categoria, AVG(monto) as promedio_venta
    FROM ventas
    GROUP BY categoria
    HAVING AVG(monto) > 100
    ORDER BY promedio_venta DESC
""")

ventas_por_categoria.show()



+-----------+------------------+----------------------+
|  categoria|      total_ventas|cantidad_transacciones|
+-----------+------------------+----------------------+
|      Hogar|522706.61980724335|                  2000|
|   Juguetes|515979.06014442444|                  2023|
|Electrónica| 513918.2698316574|                  2006|
|     Libros| 510469.2196073532|                  1986|
|       Ropa|487864.68007564545|                  1985|
+-----------+------------------+----------------------+



In [8]:
#=====================================================================
# Guardar resultados en formato Parquet para la siguiente lección.
#=====================================================================


# Tarea: Guardar resultados en formato Parquet, ocupa menos espacio que CSV
# Esto crea una carpeta optimizada lista para la Lección 5
df_retail.write.mode("overwrite").parquet("retail_data_processed.parquet")

print("Archivo Parquet generado y guardado exitosamente.")


#=====================================================================
# Entregar notebook + archivo Parquet generado.
#=====================================================================

# Comprimir la carpeta Parquet para poder descargarla desde el panel de archivos
!zip -r retail_data_final.zip retail_data_processed.parquet

Archivo Parquet generado y guardado exitosamente.
  adding: retail_data_processed.parquet/ (stored 0%)
  adding: retail_data_processed.parquet/._SUCCESS.crc (stored 0%)
  adding: retail_data_processed.parquet/.part-00000-f3f97b31-6bc4-445c-a733-d54f067d10e9-c000.snappy.parquet.crc (stored 0%)
  adding: retail_data_processed.parquet/_SUCCESS (stored 0%)
  adding: retail_data_processed.parquet/.part-00001-f3f97b31-6bc4-445c-a733-d54f067d10e9-c000.snappy.parquet.crc (stored 0%)
  adding: retail_data_processed.parquet/part-00000-f3f97b31-6bc4-445c-a733-d54f067d10e9-c000.snappy.parquet (deflated 34%)
  adding: retail_data_processed.parquet/part-00001-f3f97b31-6bc4-445c-a733-d54f067d10e9-c000.snappy.parquet (deflated 34%)


## Lección 5: Introducción a Machine Learning Escalable (Spark MLlib)


📌 Objetivo: Construir un pipeline de MLlib para clasificación y segmentación
de usuarios.

📍 Tareas a desarrollar:
- Cargar los DataFrames procesados en la etapa anterior.
- Crear features con VectorAssembler e indexar categorías con StringIndexer.
- Entrenar un modelo supervisado (Regresión Logística) y uno no supervisado (K-Means).
- Evaluar métricas y generar un reporte final con insights para marketing.
- Entregar pipeline completo en notebook + informe final en PDF.

In [13]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans

#=====================================================================
# Cargar los DataFrames procesados en la etapa anterior.
#=====================================================================
# 1. Cargar el archivo Parquet de la Lección 4
df_final = spark.read.parquet("retail_data_processed.parquet")


#=====================================================================
# Crear features con VectorAssembler e indexar categorías con StringIndexer.
#=====================================================================
# 2. Preparar las características (usaremos el 'monto' para los modelos)
assembler = VectorAssembler(inputCols=["monto"], outputCol="features")
data_ml = assembler.transform(df_final)



#=====================================================================
# Entrenar un modelo supervisado (Regresión Logística) y uno no supervisado (K-Means)
#=====================================================================
# --- MODELO A: Regresión Logística (Clasificación) ---
# Simulamos una etiqueta: supongamos que montos > 300 son "Premium" (1) y < 300 son "Normal" (0)
from pyspark.sql.functions import when
data_log = data_ml.withColumn("label", when(data_ml["monto"] > 300, 1).otherwise(0))

lr = LogisticRegression(featuresCol="features", labelCol="label")
modelo_lr = lr.fit(data_log)
print("Modelo de Regresión Logística entrenado.")

# --- MODELO B: K-Means (Clustering) ---
# Queremos agrupar a los clientes en 3 segmentos según su gasto
kmeans = KMeans(k=3, seed=1)
modelo_kmeans = kmeans.fit(data_ml)



#=====================================================================
# Evaluar métricas y generar un reporte final con insights para marketing.
#=====================================================================
# Mostrar los centros de los clusters (el gasto promedio de cada grupo)
centers = modelo_kmeans.clusterCenters()
print("\nSegmentación de Clientes (Centros de Cluster):")
for i, center in enumerate(centers):
    print(f"Cluster {i}: Gasto promedio de ${center[0]:.2f}")




#=====================================================================
# Entregar pipeline completo en notebook + informe final en PDF.
#=====================================================================
# Verificación del Pipeline Completo (Lección 5 - Resultado Final
# Ver resultados de la segmentación en los datos reales
print("\n\n")
predicciones = modelo_kmeans.transform(data_ml)
predicciones.select("id_usuario", "monto", "prediction").show(10)

print("\n")
print("0: Clientes Premium / VIP: Los usuarios que más dinero dejan en RetailMax.\n")
print("1: Clientes de Bajo Gasto: Usuarios que realizan compras pequeñas o económicas.\n")
print("2: Clientes de Gasto Medio: El grupo estándar que mantiene un consumo equilibrado.\n")


Modelo de Regresión Logística entrenado.

Segmentación de Clientes (Centros de Cluster):
Cluster 0: Gasto promedio de $417.90
Cluster 1: Gasto promedio de $89.49
Cluster 2: Gasto promedio de $253.81



+----------+------+----------+
|id_usuario| monto|prediction|
+----------+------+----------+
|   user_58|429.28|         0|
|  user_929|217.71|         2|
|  user_783|295.41|         2|
|  user_703|101.55|         1|
|  user_786|439.36|         0|
|  user_160|157.05|         1|
|  user_155|487.57|         0|
|  user_578|236.09|         2|
|  user_574|362.67|         0|
|  user_412| 59.24|         1|
+----------+------+----------+
only showing top 10 rows


0: Clientes Premium / VIP: Los usuarios que más dinero dejan en RetailMax.

1: Clientes de Bajo Gasto: Usuarios que realizan compras pequeñas o económicas.

2: Clientes de Gasto Medio: El grupo estándar que mantiene un consumo equilibrado.

